# Lab 1: Neo4j Aura Setup

This notebook sets up the Neo4j database you use for the rest of the workshop, then teaches you enough Cypher to read the queries in Labs 2 through 6.

You do two things here. First you create a free Neo4j Aura instance and save its credentials. Then you build a tiny aircraft graph by hand, query it, and delete it.

## What You'll Learn
- How to create an AuraDB Free instance and save its credentials
- How to connect to Aura from a Databricks notebook with the Neo4j Python driver
- How to create nodes and relationships with Cypher
- How to read the graph back with MATCH, RETURN, and WHERE
- How to clear the graph before Lab 2

## Prerequisites
- A valid email address
- Access to the workshop Databricks cluster (the `neo4j` Python driver is already installed on it)

## Instructions
1. Clone this notebook to your personal folder
2. Work through Part 1 in your browser, then come back
3. Enter your Neo4j credentials in the Configuration cell
4. Run the remaining cells in order (Shift+Enter)

## Part 1: Create Your AuraDB Free Instance

Neo4j Aura is Neo4j's managed cloud service. AuraDB Free is the tier this workshop uses. It costs nothing, needs no credit card, and does not expire.

Do this part in a separate browser tab, then return to this notebook with your credentials.

### Sign Up

1. Navigate to [https://console-preview.neo4j.io/](https://console-preview.neo4j.io/)

2. Click **"Don't have an account? Sign up"** below the login form.

3. Follow the sign-up process to create your Neo4j Aura account. You need to provide your email address, create a password, and agree to the terms of service.

4. When prompted "Where do you want your instance deployed?", configure the following:
   - **Cloud provider:** AWS
   - **Region:** Europe, Paris (eu-west-3)

> **WARNING:** Do **not** click **Start 14-day free trial**. That button provisions an AuraDB Professional trial instance, which expires after 14 days, partway through this course. This workshop uses AuraDB Free. Below the button, under "Not looking to start a free trial?", click **Select another instance type**.

![Instance deployment configuration showing AWS cloud provider and Europe Paris region](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/site/modules/ROOT/images/lab1-free-tier-location.png)

### Create the Instance and Save the Credentials

5. Choose the **AuraDB Free** tier, then confirm to create the instance. The confirm button is labelled **Next** or **Create instance**, depending on which version of the console you land on.

6. Your AuraDB Free instance is created automatically. A dialog appears showing a username and a password. **Save your credentials immediately**: click **Download to continue** to save the credentials file.

> **CRITICAL:** The password is shown once and is not available after you close this dialog. Download the credentials file and store it somewhere safe. You need these credentials in every later lab.

![Creating your instance screen showing credentials with download option](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/site/modules/ROOT/images/lab1-free-create-instance.png)

7. Once your instance is running, you see it in the Instances list with a "RUNNING" status and a type of **AuraDB Free**. Copy the connection string, which looks like `neo4j+s://xxxxxxxx.databases.neo4j.io`. You need it, your username `neo4j`, and the downloaded password in every later lab.

### What AuraDB Free Gives You

AuraDB Free holds up to **200,000 nodes** and **400,000 relationships**. It stays free, needs no credit card, and does not expire.

The workshop fits comfortably inside that. After finishing Labs 1 through 3 your graph holds about 21,613 nodes, which is 10.8 percent of the node cap.

Two limits worth knowing. You can create only one Free instance per account. A Free instance with no activity for 30 days is deleted, so keep using it while the workshop runs.

### What AuraDB Free Does Not Include

Graph Data Science is not available on AuraDB Free. `Lab_2_Databricks_ETL_Neo4j/02_gds_knn_aircraft.ipynb` is therefore optional and skippable: skip it unless you have your own AuraDB Professional instance. Every other notebook in the workshop runs on AuraDB Free.

## Part 2: Save and Test Your Credentials

Enter the three values from the credentials file you downloaded. Every notebook in this workshop starts with a Configuration cell shaped exactly like this one, so get used to it here.

`NEO4J_URI` is the connection string that starts with `neo4j+s://`. `NEO4J_USERNAME` is `neo4j` unless you changed it. `NEO4J_PASSWORD` is the generated password from the downloaded file.

In [ ]:
# ============================================
# CONFIGURATION - Enter your Neo4j credentials
# ============================================

NEO4J_URI = ""  # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""  # Your password from the downloaded credentials file
NEO4J_DATABASE = "neo4j"  # Neo4j database to use (Aura default is "neo4j")

# Validate configuration
if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: Please enter your Neo4j credentials above before running the notebook!")
else:
    print("Configuration ready!")
    print(f"Neo4j URI: {NEO4J_URI}")

### Test the Connection

The cell below opens a driver, calls `verify_connectivity()`, and prints the server version. A version number means the URI, username, and password are all correct.

The `neo4j` Python driver is already installed as a cluster library, so there is no `pip install` step.

Common failures:
- **`AuthError`**: the password is wrong. Re-read it from the downloaded file.
- **`ServiceUnavailable`**: the URI is wrong, or the instance is still starting. Check the console shows "RUNNING".
- **`ValueError` from this cell**: the Configuration cell above is still blank.

In [ ]:
from neo4j import GraphDatabase

if not NEO4J_URI.strip() or not NEO4J_PASSWORD.strip():
    raise ValueError(
        "NEO4J_URI and NEO4J_PASSWORD are not set. Enter your Aura credentials "
        "in the Configuration cell above, then run this cell again."
    )

with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as test_driver:
    test_driver.verify_connectivity()
    _, summary, _ = test_driver.execute_query("RETURN 1 AS ok", database_=NEO4J_DATABASE)

print("Connected to Neo4j Aura")
print(f"Server:  {summary.server.address}")
print(f"Version: {summary.server.agent}")

## Part 3: Cypher Basics

Cypher is Neo4j's query language. It draws patterns. `(a:Aircraft)` is a node, `-[:HAS_SYSTEM]->` is a relationship, and a query is those pieces joined into a shape you want to find or create.

Before you load real data in Lab 2, build a two-node graph by hand and query it.

### Two Ways to Run These Queries

Every query below runs from this notebook through the Python driver. The same text also runs unchanged in the Aura Query editor, which shows results as a picture of the graph rather than a table:

1. Go to [console.neo4j.io](https://console.neo4j.io)
2. Select your instance
3. Click **Query** to open the query editor

Run each query here first, then paste it into the Query editor to see the same result drawn as nodes and lines.

### A Helper for Running Cypher

One driver and one function, defined once and reused by every cell below. `run_cypher` sends a Cypher string to Aura, reports what changed, and returns the rows as a pandas DataFrame.

In [ ]:
import pandas as pd
from neo4j import GraphDatabase
from neo4j.graph import Node, Relationship

# One driver for the rest of the notebook. Closed in the cleanup section.
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()


def _to_cell(value):
    """Flatten graph objects into something a DataFrame column can hold."""
    if isinstance(value, Node):
        return dict(value)
    if isinstance(value, Relationship):
        return f"[:{value.type}]"
    return value


def run_cypher(query, **params):
    """Run a Cypher query and return the rows as a pandas DataFrame."""
    records, summary, keys = driver.execute_query(query, database_=NEO4J_DATABASE, **params)

    c = summary.counters
    changes = {
        "nodes created": c.nodes_created,
        "relationships created": c.relationships_created,
        "properties set": c.properties_set,
        "nodes deleted": c.nodes_deleted,
        "relationships deleted": c.relationships_deleted,
    }
    changed = ", ".join(f"{name}: {n}" for name, n in changes.items() if n)
    print(changed if changed else "no changes to the graph")

    if not records:
        print("no rows returned")
        return None
    return pd.DataFrame([[_to_cell(r[k]) for k in keys] for r in records], columns=keys)


print("Helper ready.")

### Creating Nodes

Create an Aircraft node with properties:

`CREATE` makes a node. `:Aircraft` is a **label** (like a type). Properties go inside curly braces.

In [ ]:
run_cypher("""
    CREATE (a:Aircraft {tail_number: 'N12345', model: 'B737-800', manufacturer: 'Boeing'})
    RETURN a
""")

### Reading Nodes

Find all Aircraft nodes and return their properties:

`MATCH` finds patterns in the graph. `RETURN` selects what to display.

In [ ]:
run_cypher("""
    MATCH (a:Aircraft)
    RETURN a.tail_number, a.model, a.manufacturer
""")

### Creating Relationships

Create two nodes connected by a relationship:

`-[:HAS_SYSTEM]->` creates a directed relationship from the Aircraft to the System.

This adds a second Aircraft node with the same tail number. That is fine here. Lab 2 adds uniqueness constraints so the real dataset cannot pick up duplicates.

In [ ]:
run_cypher("""
    CREATE (a:Aircraft {tail_number: 'N12345', model: 'B737-800'})
    CREATE (s:System {name: 'Engine #1', type: 'CFM56-7B'})
    CREATE (a)-[:HAS_SYSTEM]->(s)
    RETURN a, s
""")

### Querying Relationships

Traverse relationships to find connected nodes:

In [ ]:
run_cypher("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(s:System)
    RETURN a.tail_number, s.name, s.type
""")

### Filtering with WHERE

Add conditions to narrow results:

In [ ]:
run_cypher("""
    MATCH (a:Aircraft)
    WHERE a.manufacturer = 'Boeing'
    RETURN a.tail_number, a.model
""")

## Cleanup Before Lab 2

Remove all nodes and relationships to start fresh:

`DETACH DELETE` removes nodes and all their relationships. Plain `DELETE` refuses to remove a node that still has relationships attached, so `DETACH` is what makes this one line enough.

> **Warning:** Run this cleanup before starting Lab 2. Lab 2 expects an empty graph and loads the full Aircraft Digital Twin dataset from scratch. Practice nodes left behind show up in its verification counts.

> **Tip:** These examples are for learning. In Lab 2 you load the full dataset programmatically using the Neo4j Spark Connector.

In [ ]:
run_cypher("MATCH (n) DETACH DELETE n")

# Confirm the graph is empty
print(run_cypher("MATCH (n) RETURN count(n) AS remaining"))

# Done with the driver. Re-run the helper cell above to reopen it.
driver.close()

## Next Steps

Your Aura instance is running and empty, and you have its credentials saved.

Continue to **Lab 2 - Databricks ETL to Neo4j** to load the Aircraft Digital Twin dataset into this instance with the Neo4j Spark Connector.

Keep the credentials file handy. Every notebook from here on starts with the same Configuration cell you filled in above.